# 02 (Kaggle) — M0 retrain (clean slate), diagnostics, M1

**The original M0 is unusable** -- its training loss climbed to 3.5+ well
before it ever went NaN, and finished at loss 4.6 (vs. M1's 0.9), a real
training failure independent of the `generate.py` quantization bug that
was separately fixed. This notebook trains a **fresh** M0 from step 0 at
`configs/m0_retrain_kaggle.yaml` (lr=1e-4 from the start -- M1's settings,
which had zero NaN events across all 601 steps), not a resume of the old
run.

**Before running -- this matters more than usual here:**
1. Attach `verilog-slm-data` (`corpus.jsonl`, eval sets).
2. Do **NOT** attach the old `verilog-slm-ckpt` dataset (the broken run's
   checkpoints) in this notebook, ever -- the bootstrap cell below
   auto-restores any `checkpoint-*/` it finds in any attached dataset, and
   if it finds the old broken checkpoint-600 it will resume from it instead
   of starting fresh, silently reintroducing the exact problem this retrain
   exists to fix.
3. For a *second or later* session of this same retrain (once it's saved
   its own checkpoints), create a **new** dataset for them --
   e.g. `verilog-slm-m0-retrain-ckpt` -- and attach only that one, never
   the old `verilog-slm-ckpt`.
4. Notebook settings: Accelerator = **GPU T4 x2**, **Internet = On**.

At the end of the session (see the cell below the training cell): Quick
Save, then push `artifacts/m0/checkpoint-<N>/` as a new version of your
*new* retrain-checkpoint dataset -- not the old `verilog-slm-ckpt`.

In [ ]:
# --- Kaggle-native bootstrap (replaces Colab's Drive-mount cell) ---
import os, shutil, glob

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/verilog-slm'):
    os.system(f'git clone {REPO} /kaggle/working/verilog-slm')
os.chdir('/kaggle/working/verilog-slm')
os.system('git pull')

os.makedirs('artifacts', exist_ok=True)
os.makedirs('data/eval', exist_ok=True)
os.makedirs('artifacts/m0', exist_ok=True)

# Search all of /kaggle/input rather than building a path from a dataset
# slug: Kaggle's actual mount point for an attached dataset has been seen
# both as /kaggle/input/<slug>/ and /kaggle/input/datasets/<user>/<slug>/,
# and isn't worth depending on -- filename/dirname matching already
# disambiguates regardless of nesting depth or which convention applies.
def _find(filename):
    for p in glob.glob(f'/kaggle/input/**/{filename}', recursive=True):
        return p
    return None

for fname, dest in [
    ('corpus.jsonl', 'artifacts/corpus.jsonl'),
    ('verilogeval_v2.jsonl', 'data/eval/verilogeval_v2.jsonl'),
    ('rtllm_v2.jsonl', 'data/eval/rtllm_v2.jsonl'),
]:
    src = _find(fname)
    if src:
        shutil.copy(src, dest)
        print(f'restored {dest} <- {src}')
    else:
        print(f'WARNING: {fname} not found under /kaggle/input')

# Match on adapter_config.json's presence, not just a "checkpoint-*"
# directory name -- a zip upload can land double-nested (the zip's own
# internal checkpoint-400/ folder landing inside another checkpoint-400/
# wrapper from the upload), and name-only matching finds the empty outer
# wrapper first, which then blocks the real inner one via the
# already-exists check below. A directory containing adapter_config.json
# is unambiguously a real checkpoint regardless of how deep it's nested.
ckpt_dirs = sorted({
    os.path.dirname(p)
    for p in glob.glob('/kaggle/input/**/checkpoint-*/adapter_config.json', recursive=True)
})
if not ckpt_dirs:
    print('WARNING: no checkpoint-*/adapter_config.json found under /kaggle/input -- starting M0 from scratch')
for ckpt_dir in ckpt_dirs:
    dest = f"artifacts/m0/{os.path.basename(ckpt_dir)}"
    if os.path.exists(dest):
        print(f'skipped {dest} <- {ckpt_dir} (destination already exists -- '
              f'rm -rf it first if you want to force a fresh copy)')
    else:
        shutil.copytree(ckpt_dir, dest)
        print(f'restored {dest} <- {ckpt_dir}')


In [ ]:
# Pinned deps, same as the Colab notebook.
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())
!nvidia-smi -L

In [ ]:
# RTL toolchain: iverilog required, yosys/verible optional (soft-gated --
# see docs/industry_standards.md). Same install as the Colab notebook.
import os
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

## Train M0, clean slate

`configs/m0_retrain_kaggle.yaml` extends `m0_baseline_kaggle.yaml` (same
`run_name: m0`, same `max_wall_hours: 11.5` wall-clock guard) but sets
`lr: 1.0e-4` from step 0, instead of the original's `base.yaml`-inherited
2e-4 that was never actually what M0 finished on. If the bootstrap cell
above printed any `restored artifacts/m0/checkpoint-...` line, **stop and
check where it came from** (`find /kaggle/input -iname adapter_config.json`)
before running the cell below -- it means something under
`/kaggle/input` had old checkpoints in it, and this would resume from them
instead of starting fresh.

In [ ]:
!python -m src.train.sft --config configs/m0_retrain_kaggle.yaml 2>&1 | tee artifacts/m0_stdout.log

## End of session -- carry the checkpoint forward

1. Confirm the run stopped on a clean checkpoint: last line of the cell
   above should be either the wall-clock-budget message or full completion,
   not a truncated stack trace.
2. **Quick Save** (not "Save & Run All").
3. Push `artifacts/m0/checkpoint-<N>/` as a new version of your **new**
   retrain-checkpoint dataset (e.g. `verilog-slm-m0-retrain-ckpt`) --
   never the old `verilog-slm-ckpt`.
4. Next session: attach `verilog-slm-data` + your new retrain-checkpoint
   dataset (still never the old one), re-run from the top -- the
   bootstrap cell restores the new checkpoint and training resumes from
   there.

## Diagnostic pass -- produces the table M1's reweighting needs

Cheap (5 samples/problem, inference not training). Runs directly against
`artifacts/m0/final` -- already local in this session if you just
finalized M0 here; otherwise see notebooks/03_diagnose.ipynb for the
Colab version of the fuller classifier-validation + taxonomy-table
breakdown, not yet ported to Kaggle.

In [ ]:
!python -m src.infer.generate --adapter artifacts/m0/final --split probe \
    --n 5 --temperature 0.8 --top_p 0.95 --out artifacts/m0_probe_gens.jsonl
!python -m src.eval.diagnose --gens artifacts/m0_probe_gens.jsonl --out artifacts/m0_diagnostic.json

## M1 -- curriculum SFT, same step budget as M0

`assert_matches_m0` inside `sft.py` fails loudly if the base checkpoint
hash or step count diverge from M0's `run_meta.json` -- this is what
keeps the ablation clean. Since M0 was finalized early (see
`artifacts/m0/run_meta.json`'s `stopped_early_reason`), M1 will also
train for that same (reduced) step count, inherited automatically.

M1 does **not** need M0's trained adapter weights -- it starts a fresh
LoRA init from the same base checkpoint, differing from M0 only in the
sampler (flat vs. curriculum). It only needs `artifacts/m0/run_meta.json`
(for the assert) and `artifacts/m0_diagnostic.json` (for the curriculum
reweighting), both already local in this session if you're continuing
directly from the cells above.

If you're instead starting a **fresh session** to resume M1's own
training later, the bootstrap cell above won't restore these two m0
files (or any `artifacts/m1/checkpoint-N/`) automatically yet -- upload
them to a Kaggle Dataset the same way as M0's checkpoints, and use the
same `find /kaggle/input -iname ...` + manual `cp` pattern from M0's
recovery if the bootstrap's generic search doesn't pick them up
correctly (it searched by `checkpoint-*/adapter_config.json` and
`corpus.jsonl`/eval-set filenames only -- `run_meta.json` and
`m0_diagnostic.json` aren't in that list, so restore them manually:
`cp /kaggle/input/<dataset>/run_meta.json artifacts/m0/` and similarly
for `m0_diagnostic.json` into `artifacts/`).

In [ ]:
!python -m src.train.sft --config configs/m1_curriculum_kaggle.yaml 2>&1 | tee -a artifacts/m1_stdout.log

In [ ]:
# Plot realised category histogram: M1 actually saw vs. M0 (uniform) --
# the direct evidence the curriculum did what it was designed to do.
import json, matplotlib.pyplot as plt
m1_meta = json.load(open('artifacts/m1/run_meta.json'))
hist = m1_meta['realised_histogram']['construct']
plt.bar(hist.keys(), hist.values())
plt.xticks(rotation=60, ha='right')
plt.title('M1 realised construct-tag exposure over training')
plt.tight_layout()
plt.savefig('artifacts/m1_realised_histogram.png')
plt.show()

## End of M1 session -- same pattern as M0

1. Confirm the run stopped cleanly (wall-clock message, full completion,
   or a deliberate finalize-early decision like M0's).
2. **Quick Save** (not "Save & Run All").
3. Push `artifacts/m1/` (+ `run_meta.json` once M1 is finalized, same
   `scripts/finalize_m0.py` pattern works for M1 -- pass its checkpoint
   and `configs/m1_curriculum_kaggle.yaml`) as a new Kaggle Dataset
   version, separate from M0's.